# 01 — API download and sample

Download a subset of the ADEME DPE Logements Existants dataset (post-July 2021) and save it locally to `data/raw/`. Downstream notebooks read from the save parquet so they never re-call the API.

- Server-side filter: `type_batiment in {maison, appartement}` (drops whole-building `immeuble` entries).
- Page size 10k.
- Set `FORCE_REFRESH = True` only when we want to re-download from ADEME.

In [1]:
import pandas as pd
import requests
from pathlib import Path
from time import sleep

In [2]:
BASE_URL = "https://data.ademe.fr/data-fair/api/v1/datasets/dpe03existant/lines"
PAGE_SIZE = 10000
TARGET_ROWS = 50000
RAW_DIR = Path("../data/raw")
RAW_PATH = RAW_DIR / f"dpe_logements_{TARGET_ROWS}.parquet"
FORCE_REFRESH = False

QS_FILTER = 'type_batiment:"maison" OR type_batiment:"appartement"'


def fetch_logements(target_rows=TARGET_ROWS, page_size=PAGE_SIZE, qs=QS_FILTER):
    params = {"size": page_size, "qs": qs}
    url = BASE_URL
    pages = []
    fetched = 0
    while url and fetched < target_rows:
        r = requests.get(url, params=params if url == BASE_URL else None, timeout=60)
        r.raise_for_status()
        payload = r.json()
        results = payload.get("results", [])
        if not results:
            break
        pages.append(pd.DataFrame(results))
        fetched += len(results)
        print(f"fetched {fetched:>7} / {target_rows} (total available: {payload.get('total')})")
        url = payload.get("next")
        sleep(0.2)
    return pd.concat(pages, ignore_index=True).head(target_rows)


if RAW_PATH.exists() and not FORCE_REFRESH:
    df = pd.read_parquet(RAW_PATH)
    print(f"loaded cached raw data from {RAW_PATH} ({RAW_PATH.stat().st_size / 1e6:.1f} MB)")
else:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    df = fetch_logements()
    df.to_parquet(RAW_PATH, index=False)
    print(f"saved raw data to {RAW_PATH} ({RAW_PATH.stat().st_size / 1e6:.1f} MB)")

df.shape

loaded cached raw data from ../data/raw/dpe_logements_50000.parquet (23.9 MB)


(50000, 220)

In [3]:
print("rows:", len(df))
print("cols:", df.shape[1])
print()
print("type_batiment counts:")
print(df["type_batiment"].value_counts(dropna=False))

rows: 50000
cols: 220

type_batiment counts:
type_batiment
appartement    33544
maison         16456
Name: count, dtype: int64
